# 08 Export Decoder And Cpu Benchmark

Purpose: execute the 08 export decoder and cpu benchmark stage.
Required inputs: DATA_ROOT plus artifacts from prior stages when listed.
Expected outputs: decoder_state_dict.pt and resource_metrics.json.
Fold: FOLD; seed: SEED; all unexecuted results remain `not_run`.


In [ ]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLD = 1
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
# This stage consumes prepared arrays/checkpoints; DATA_ROOT is not required.


In [ ]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))

# Separate Kaggle jobs do not share their writable files automatically.
# If prior artifacts are mounted under /kaggle/input, set this path explicitly.
ARTIFACT_INPUT_ROOT = os.environ.get('PPGCVAE_INPUT_ROOT')
if ARTIFACT_INPUT_ROOT:
    import shutil
    source_root = Path(ARTIFACT_INPUT_ROOT)
    if not source_root.is_dir(): raise FileNotFoundError(source_root)
    if source_root.resolve() != OUTPUT_ROOT.resolve():
        OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
        for folder in ('prepared','proxy','tuning','cvae','gaussian','ablations'):
            source = source_root/folder
            if source.is_dir(): shutil.copytree(source,OUTPUT_ROOT/folder,dirs_exist_ok=True)
    print('Imported previous artifacts from:', source_root)


In [ ]:
from ppg_cvae.models.cvae import ConditionalVAE
from ppg_cvae.models.hr_proxy import HRProxy
from ppg_cvae.evaluation.resources import measure_resources
from ppg_cvae.utils.io import load_yaml, save_json, environment_info
run_dir = OUTPUT_ROOT/'cvae'/f'fold_{FOLD:02d}'/f'seed_{SEED}'
config = load_yaml(run_dir/'resolved_config.yaml')
model = ConditionalVAE(config['latent_dim'])
model.load_state_dict(torch.load(run_dir/'best_model.pt', map_location='cpu', weights_only=True))
proxy = HRProxy()
proxy.load_state_dict(torch.load(OUTPUT_ROOT/'proxy'/f'fold_{FOLD:02d}'/'hr_proxy.pt', map_location='cpu', weights_only=True))
torch.save(model.decoder.state_dict(), run_dir/'decoder_state_dict.pt')
metrics = measure_resources(model, proxy, run_dir/'decoder_state_dict.pt')
metrics['environment'] = environment_info()
metrics['checkpoint'] = str(run_dir/'best_model.pt')
save_json(metrics, run_dir/'resource_metrics.json')
print(metrics)
